# Mean Squared Error (MSE) and R² for Regression Evaluation

After training a regression model, you must answer two critical questions:

How large are the prediction errors?
How much of the target's variation does the model explain?

Mean Squared Error (MSE) and R² answer those questions from fundamentally different perspectives.

MSE measures average squared prediction error, an absolute quantity in the scale of your target.
R² measures how much variance in the target the model explains relative to the simplest possible baseline, which is always predicting the mean.

Neither metric alone tells the full story. MSE without R² gives no sense of improvement over doing nothing. R² without MSE gives no sense of how large the actual errors are. Together, they provide both error magnitude insight and relative explanatory power.

This lesson covers:

- What MSE is and how it is calculated
- Why squaring errors fundamentally changes evaluation behavior
- What R² means mathematically and practically
- How R² relates to the mean baseline
- How to compute both metrics in scikit-learn
- How to interpret them together, including when they tell contradictory stories
- Common pitfalls that lead to misinterpretation

## Mean Squared Error (MSE)

Mean Squared Error is the average of squared differences between predicted and true values:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

Where:

- `y_i` is the actual observed value
- `\hat{y}_i` is the model's predicted value
- `n` is the number of samples

Every prediction error is squared before averaging. That single decision has profound consequences for how MSE behaves.

### Why Squared Error?

Squaring errors does two things simultaneously:

- It makes all errors positive, so over-predictions and under-predictions both contribute positively, with no cancellation.
- It penalizes large errors disproportionately. An error of 10 contributes 100 to MSE, not 10. An error of 20 contributes 400.

This sensitivity is a feature, not a bug, when large errors are genuinely more costly than small ones.

## Units of MSE

Because errors are squared, MSE inherits squared units:

- If your target is measured in lakhs, MSE is in lakhs²
- If your target is measured in hours, MSE is in hours²
- If your target is measured in °C, MSE is in °C²

These squared units have no physical or business meaning, which makes MSE difficult to communicate directly. That is why practitioners often prefer RMSE for reporting:

$$
\text{RMSE} = \sqrt{\text{MSE}}
$$

RMSE restores the original units while retaining MSE's sensitivity to large errors. Use MSE internally for optimization and model comparison. Use RMSE when communicating results to stakeholders.

### When to Use MSE

MSE is the right choice when:

- Large errors are especially harmful
- You want to penalize outlier predictions strongly
- The model is optimized using squared loss
- You are comparing models mathematically and want a differentiable objective

## What Is R² (Coefficient of Determination)?

R² measures how much of the total variation in the target variable is explained by your model, relative to the simplest possible baseline: always predicting the mean.

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Where:

- `SS_res` is the residual sum of squares, the model's total squared error
- `SS_tot` is the total sum of squares, the squared error of always predicting the mean

The ratio tells you what fraction of the target's variance your model failed to explain. Subtracting from 1 gives you what fraction it did explain.

### Interpreting R²

| R² value | Meaning |
| --- | --- |
| 1.0 | Perfect prediction |
| 0.75 | Model explains 75% of target variance |
| 0.0 | Model performs identically to the mean baseline |
| < 0 | Model is worse than always predicting the mean |

R² is not a percentage accuracy. It is a proportion of explained variance.

## Relationship Between MSE and R²

MSE and R² are related, but they measure fundamentally different things:

- MSE is an absolute quantity, the raw magnitude of squared errors in squared target units.
- R² is a relative quantity, improvement over the mean baseline expressed as a proportion of total variance.

This distinction matters because the two metrics can move in opposite directions depending on the data:

### Low MSE, Low R²

Your model's errors may be small in absolute terms, but the target itself may have very low variance. The baseline also makes small errors, so the model's relative improvement is modest.

### High MSE, High R²

The target values may span a large range. Even with large absolute errors, the model can dramatically outperform the mean baseline. R² reveals strong relative performance even when MSE looks alarming.

The key principle is simple: neither metric contextualizes itself. MSE needs a baseline to be interpretable. R² needs target scale and business tolerance to be actionable. Always supply both.

## Implementing MSE and R² in Scikit-Learn

```python
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")
```

## Comparing Against Baseline

Always compare against the mean baseline. That baseline is the definition of R² = 0.

```python
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:25s} | MSE: {mse:8.2f} | RMSE: {rmse:6.2f} | R²: {r2:.3f}")

evaluate("Baseline (mean)", y_test, baseline_preds)
evaluate("Linear Regression", y_test, model_preds)
```

Why baseline R² is approximately 0: by definition, the squared error of the mean predictor equals the total sum of squares, so the baseline explains none of the variance.

## Interpreting MSE and R² Together

Example A - Strong improvement:

| Model | MSE | RMSE | R² |
| --- | --- | --- | --- |
| Baseline | 400 | 20.0 | 0.000 |
| Model | 100 | 10.0 | 0.750 |

The model cuts squared error by 75% and halves the RMSE. R² = 0.75 confirms it explains three-quarters of the target's variance. This is a clear, meaningful improvement.

Example B - Marginal improvement:

| Model | MSE | RMSE | R² |
| --- | --- | --- | --- |
| Baseline | 400 | 20.0 | 0.000 |
| Model | 390 | 19.7 | 0.025 |

Technically better than baseline, but R² = 0.025 means the model explains just 2.5% of variance. The right response is better feature engineering, not a more complex model.

Example C - Negative R²:

| Model | MSE | RMSE | R² |
| --- | --- | --- | --- |
| Baseline | 400 | 20.0 | 0.000 |
| Model | 520 | 22.8 | -0.300 |

The model is worse than predicting the mean for everyone. Investigate data leakage, train/test shift, bugs in the prediction pipeline, or severe overfitting.

## Cross-Validation for Stability

A single test split can be misleading. Cross-validation gives you mean performance and variance across held-out folds.

```python
from sklearn.model_selection import cross_val_score
import numpy as np

cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print(f"CV R² scores: {cv_r2.round(3)}")
print(f"Mean CV R²:   {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

cv_mse = -cross_val_score(
    model, X_train, y_train, cv=5, scoring="neg_mean_squared_error"
 )
cv_rmse = np.sqrt(cv_mse)
print(f"CV RMSE scores: {cv_rmse.round(3)}")
print(f"Mean CV RMSE:   {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")
```

What stable results look like: low standard deviation across folds. Watch for negative R² in individual folds, because that means the model is failing entirely on those subsets.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

# Built-in regression dataset with a real-valued target
X, y = load_diabetes(return_X_y=True, as_frame=True)

# Split first so the test set stays untouched until the end
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
 )

# Baseline: predict the training mean
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

# Model: ordinary least squares regression
model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

baseline_mse = mean_squared_error(y_test, baseline_preds)
model_mse = mean_squared_error(y_test, model_preds)
baseline_rmse = np.sqrt(baseline_mse)
model_rmse = np.sqrt(model_mse)
baseline_r2 = r2_score(y_test, baseline_preds)
model_r2 = r2_score(y_test, model_preds)

mse_reduction = baseline_mse - model_mse
pct_improve = (mse_reduction / baseline_mse) * 100
residuals = model_preds - y_test.to_numpy()

def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:25s} | MSE: {mse:8.2f} | RMSE: {rmse:6.2f} | R^2: {r2:.3f}")

evaluate("Baseline (mean)", y_test, baseline_preds)
evaluate("Linear Regression", y_test, model_preds)
print(f"MSE reduction vs baseline: {mse_reduction:.2f} ({pct_improve:.1f}%)")
print(f"Baseline R^2: {baseline_r2:.3f}")
print(f"Residual mean (bias check): {residuals.mean():.2f}")

cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
cv_mse = -cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
cv_rmse = np.sqrt(cv_mse)

print(f"CV R^2 scores: {cv_r2.round(3)}")
print(f"Mean CV R^2:   {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")
print(f"CV RMSE scores: {cv_rmse.round(3)}")
print(f"Mean CV RMSE:   {cv_rmse.mean():.3f} +/- {cv_rmse.std():.3f}")

## When MSE and R² Tell Different Stories

This happens often enough that it deserves its own section.

Low MSE but low R²: the target variance is small, so even the baseline makes small errors. Your model's absolute errors are small, but the baseline is already strong, so relative improvement is modest.

High MSE but high R²: the target spans a large range. Absolute errors appear large, but the baseline's errors are even larger. R² correctly identifies strong relative performance.

R² positive but MSE suspiciously low can point to data leakage. If both metrics look better than expected, verify your train/test split and feature pipeline.

## Common Pitfalls to Avoid

- Ignoring the baseline
- Reporting R² without explaining what it measures
- Comparing MSE across datasets with different target scales
- Missing negative R², which is a critical failure signal
- Using training metrics instead of test metrics
- Ignoring cross-validation variance

## Practical Checklist Before Reporting

- MSE and R² computed on test data only
- Both compared against the mean baseline
- R² interpreted as proportion of explained variance, not percentage accuracy
- RMSE reported alongside MSE for interpretability
- Cross-validation run, with mean and std reported
- Negative R² investigated if present
- MSE interpreted relative to target scale and business tolerance
- Both metrics reported together, not in isolation

## Closing Reflection

MSE tells you how large your squared errors are. R² tells you how much better you are than predicting the mean. One measures magnitude. The other measures relative explanatory power. They are complements, not substitutes.

A model with low MSE and high R² is doing well on both fronts. A model with one metric looking good and the other looking bad is telling you something important.

Never report one without understanding the other.
Never report either without a baseline.
Never interpret numbers without context.


## Bonus Content

- Scikit-learn Model Evaluation Guide
- Regression Metrics - Scikit-learn